# Day-ahead consumption forecast

Forecast national consumption for every hour of day D, with the decision taken at 12:00
on D-1. Features are restricted to what is known at 12:00 D-1: lagged consumption, calendar
terms, and the temperature forecast issued at or before 12:00 D-1.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.width", 120)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time")
fc = pd.read_csv("../data/weather_forecasts.csv", parse_dates=["origin_datetime", "forecast_datetime"])
assert df["time"].is_unique and df["time"].is_monotonic_increasing
print(df.shape, fc.shape, df["time"].min(), df["time"].max())

(17520, 6) (70004, 4) 2022-01-01 00:00:00+00:00 2023-12-31 23:00:00+00:00


## Target and decision time

Target: consumption at hour *t* of day D. Decision time for that hour: 12:00 on D-1.

In [2]:
s = df.set_index("time")["consumption_mwh"]
frame = pd.DataFrame({"y": s})
frame["decision_time"] = (frame.index.normalize() - pd.Timedelta(days=1) + pd.Timedelta(hours=12))
frame["lead_h"] = (frame.index - frame["decision_time"]) / pd.Timedelta(hours=1)
print(frame["lead_h"].describe()[["min", "max"]].to_dict())

{'min': 12.0, 'max': 35.0}


## Lagged consumption features

At 12:00 D-1 the latest complete day is D-2, so lags are measured from D-2 (48 h) and the
same weekday a week earlier (168 h). The 24-hour rolling mean is shifted so it ends at 12:00
D-1 at the latest.

In [3]:
frame["lag48"] = s.shift(48)
frame["lag168"] = s.shift(168)
frame["lag336"] = s.shift(336)
roll = s.shift(1).rolling(24).mean()                       # mean of the 24 h ending one hour before each stamp
frame["roll24_at_decision"] = roll.reindex(frame["decision_time"]).to_numpy()
frame["diff_week"] = frame["lag168"] - frame["lag336"]
frame[["y", "lag48", "lag168", "roll24_at_decision"]].tail(3)

,y,lag48,lag168,roll24_at_decision
time,,,,
2023-12-31 21:00:00+00:00,30263.2,33593.5,31243.1,31767.945833
2023-12-31 22:00:00+00:00,28415.8,31623.9,29816.4,31767.945833
2023-12-31 23:00:00+00:00,27054.8,30595.6,27844.9,31767.945833


## Temperature forecast known at decision time

For each target hour use the most recent forecast whose `origin_datetime` is at or before
the decision time and whose `forecast_datetime` is the target hour.

In [4]:
fc_sorted = fc.sort_values("origin_datetime")
tgt = frame.reset_index().rename(columns={"time": "target_time"})[["target_time", "decision_time"]]
tgt = tgt.sort_values("decision_time")
joined = pd.merge_asof(tgt, fc_sorted, left_on="decision_time", right_on="origin_datetime",
                       by=None, direction="backward", left_by="target_time", right_by="forecast_datetime")
joined = joined.set_index("target_time").sort_index()
has_fc = joined["origin_datetime"].notna()
assert (joined.loc[has_fc, "origin_datetime"] <= joined.loc[has_fc, "decision_time"]).all()
frame["temp_fc"] = joined["temp_forecast_c"].reindex(frame.index).to_numpy()
frame["temp_fc"] = frame["temp_fc"].ffill()
print("rows without any forecast before the decision time:", int((~has_fc).sum()), "(first day of the archive)")
print("forecast horizon used (h):", joined["horizon_h"].min(), "-", joined["horizon_h"].max())
print("missing forecast rows after ffill:", frame["temp_fc"].isna().sum())

rows without any forecast before the decision time: 24 (first day of the archive)
forecast horizon used (h): 12.0 - 35.0
missing forecast rows after ffill: 24


## Calendar features

In [5]:
frame["hdd"] = np.clip(15 - frame["temp_fc"], 0, None)
frame["cdd"] = np.clip(frame["temp_fc"] - 22, 0, None)
frame["is_weekend"] = (frame.index.dayofweek >= 5).astype(float)
hour_dummies = pd.get_dummies(frame.index.hour, prefix="h", drop_first=True).astype(float)
hour_dummies.index = frame.index
frame = pd.concat([frame, hour_dummies], axis=1)
features = ["lag48", "lag168", "lag336", "roll24_at_decision", "diff_week", "temp_fc", "hdd", "cdd", "is_weekend"] + list(hour_dummies.columns)
data = frame.dropna(subset=features + ["y"])
print(len(frame), "->", len(data), "rows after dropping warm-up NaNs")

17520 -> 17184 rows after dropping warm-up NaNs


## Chronological split

In [6]:
split_time = pd.Timestamp("2023-08-01", tz="UTC")
train = data[data.index < split_time]
test = data[data.index >= split_time]
print("train:", train.index.min().date(), "to", train.index.max().date(), len(train))
print("test :", test.index.min().date(), "to", test.index.max().date(), len(test))

train: 2022-01-15 to 2023-07-31 13512
test : 2023-08-01 to 2023-12-31 3672


## Baselines and model

In [7]:
def metrics(y, p):
    return {"RMSE": np.sqrt(mean_squared_error(y, p)), "MAE": mean_absolute_error(y, p), "R2": r2_score(y, p)}

naive = s.shift(24).reindex(test.index)
weekly = test["lag168"]
results = {"naive same hour yesterday": metrics(test["y"], naive), "same hour last week": metrics(test["y"], weekly)}

model = make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=0))
model.fit(train[features], train["y"])
pred = pd.Series(model.predict(test[features]), index=test.index)
results["ridge"] = metrics(test["y"], pred)
pd.DataFrame(results).T.round(3)

,RMSE,MAE,R2
naive same hour yesterday,1617.828,1272.509,0.833
same hour last week,1405.158,1099.668,0.874
ridge,898.677,710.706,0.948


## Coefficients and residuals

In [8]:
coef = pd.Series(model.named_steps["ridge"].coef_, index=features)
print(coef.drop(hour_dummies.columns).sort_values().round(0))
resid = test["y"] - pred
print("residual mean:", round(resid.mean(), 1), " lag-1 autocorr:", round(resid.autocorr(1), 3))
resid.groupby(resid.index.hour).mean().round(0).to_frame("bias_by_hour").T

is_weekend            -868.0
temp_fc               -126.0
lag48                  -18.0
diff_week              -15.0
cdd                     28.0
roll24_at_decision     235.0
lag168                 291.0
lag336                 297.0
hdd                   1548.0
dtype: float64
residual mean: -22.5  lag-1 autocorr: 0.618


time,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
bias_by_hour,-95.0,-1.0,-46.0,-44.0,-46.0,10.0,-22.0,24.0,-14.0,36.0,...,13.0,-68.0,-68.0,7.0,-5.0,57.0,-86.0,12.0,-44.0,-45.0


## Results

In [9]:
out = pd.DataFrame(results).T
print(out.round(1).to_string())
print()
print(f"Ridge RMSE {out.loc['ridge', 'RMSE']:,.0f} MWh vs naive {out.loc['naive same hour yesterday', 'RMSE']:,.0f} MWh "
      f"({1 - out.loc['ridge', 'RMSE'] / out.loc['naive same hour yesterday', 'RMSE']:.0%} improvement); "
      f"R2 {out.loc['ridge', 'R2']:.3f} on Aug-Dec 2023.")

                             RMSE     MAE   R2
naive same hour yesterday  1617.8  1272.5  0.8
same hour last week        1405.2  1099.7  0.9
ridge                       898.7   710.7  0.9

Ridge RMSE 899 MWh vs naive 1,618 MWh (44% improvement); R2 0.948 on Aug-Dec 2023.
